# Import des librairies

In [240]:
from dataclasses import dataclass
import re
from collections import defaultdict
import math

# Classe

In [241]:
@dataclass
class Assign:
    num_bot: int
    value: int

In [242]:
@dataclass
class Destination:
    num: int
    type: str

@dataclass
class Give:
    num_bot: int
    destination_low: Destination
    destination_high: Destination

In [243]:
@dataclass
class Bot:
    num_bot: int
    values: list
    # _min: int = None
    # _max: int = None

    # @property
    # def min(self):
    #     if self.values != []:
    #         self._min = min(self.values)
    #         return self._min

    # @property
    # def max(self):
    #     if self.values != []:
    #         self._max = max(self.values)
    #         return self._max

# Lecture du fichier source

In [244]:
PATTERN_ASSIGN = r"value (\d+) goes to bot (\d+)"
PATTERN_GIVE = r"bot (\d+) gives low to (bot|output) (\d+) and high to (bot|output) (\d+)"

In [245]:
def func_clean_row(
    in_row: str
) -> Assign | Give:

    # Si la ligne commence par "value"
    if in_row.startswith('value'):

        value_str, num_bot_str = re.search(PATTERN_ASSIGN, in_row).groups()
        
        assign = Assign(
            num_bot = int(num_bot_str),
            value = int(value_str)
        )

        return assign
    
    # Sinon
    else:
        num_bot_str, destination_low_type, destination_low_num_str, destination_high_type, destination_high_num_str = re.search(PATTERN_GIVE, in_row).groups()

        destination_low = Destination(
            num = int(destination_low_num_str),
            type = destination_low_type
        )

        destination_high = Destination(
            num = int(destination_high_num_str),
            type = destination_high_type
        )

        give = Give(
            num_bot = int(num_bot_str),
            destination_low = destination_low,
            destination_high = destination_high
        )
        
        return give

In [246]:
with open("real.txt") as fichier:
    instructions = [func_clean_row(row) for row in fichier.read().splitlines()]
    
instructions

[Give(num_bot=153, destination_low=Destination(num=105, type='bot'), destination_high=Destination(num=10, type='bot')),
 Give(num_bot=84, destination_low=Destination(num=149, type='bot'), destination_high=Destination(num=198, type='bot')),
 Give(num_bot=32, destination_low=Destination(num=124, type='bot'), destination_high=Destination(num=76, type='bot')),
 Give(num_bot=177, destination_low=Destination(num=8, type='output'), destination_high=Destination(num=19, type='bot')),
 Give(num_bot=64, destination_low=Destination(num=6, type='bot'), destination_high=Destination(num=179, type='bot')),
 Give(num_bot=115, destination_low=Destination(num=181, type='bot'), destination_high=Destination(num=190, type='bot')),
 Give(num_bot=169, destination_low=Destination(num=21, type='bot'), destination_high=Destination(num=60, type='bot')),
 Give(num_bot=106, destination_low=Destination(num=17, type='bot'), destination_high=Destination(num=84, type='bot')),
 Give(num_bot=129, destination_low=Destinat

In [247]:
list_assignations = [e for e in instructions if isinstance(e, Assign)]
list_transferts = [e for e in instructions if isinstance(e, Give)]

# Traitement des Assignation

In [248]:
dict_bot = defaultdict(list)

for assignation in list_assignations:
    dict_bot[assignation.num_bot].append(assignation.value)

dict_bot

defaultdict(list,
            {124: [11],
             93: [2],
             107: [47],
             205: [67],
             21: [71],
             120: [23],
             189: [17],
             13: [5],
             111: [3],
             30: [73],
             185: [29, 19],
             39: [59],
             169: [13],
             197: [7],
             156: [53],
             32: [43],
             18: [37],
             199: [31],
             160: [61],
             194: [41]})

In [249]:
dict_output = defaultdict()

# Traitement des transferts 

In [250]:
transferts = list_transferts.copy()

while transferts:

    # On prends le dernier element de la liste des transferts
    give = transferts.pop()

    values_bot_giver = dict_bot[give.num_bot]
    print(give.num_bot)
    print(values_bot_giver)
    print()

    # Si le robot qui doit donner, a moins de 2 values
    if len(values_bot_giver) < 2:
        
        # On remet le transfert au debut de la liste (afin de le traiter plus tard)
        transferts.insert(0,give)
        continue

    # Condition d'arret finale
    if (61 in values_bot_giver) and (17 in values_bot_giver):
        print(f"Bot {give.num_bot}") 
        

    min_bot = min(values_bot_giver)
    max_bot = max(values_bot_giver)

    # On retire les deux puces du robot
    dict_bot[give.num_bot] = []

    # On redistribue les puces
    # Si le transfert de la valeur minimale se fait vers un autre bot
    if give.destination_low.type == 'bot':
        dict_bot[give.destination_low.num].append(min_bot)
    
    # Si le transfert de la valeur minimale se fait vers un output
    else:
        print(give.destination_low)
        dict_output[give.destination_low.num] = min_bot
    
    # Si le transfert de la valeur maximale se fait vers un autre bot
    if give.destination_high.type == 'bot':
        dict_bot[give.destination_high.num].append(max_bot)

    # Si le transfert de la valeur maximale se fait vers un output
    else:
        dict_output[give.destination_high.num] = max_bot
    

# print(dict_bot)
# print(dict_output)

101
[]

149
[]

137
[]

53
[]

143
[]

108
[]

138
[]

87
[]

109
[]

67
[]

63
[]

80
[]

144
[]

168
[]

119
[]

43
[]

150
[]

40
[]

171
[]

82
[]

10
[]

188
[]

21
[71]

35
[]

102
[]

76
[]

200
[]

100
[]

41
[]

1
[]

159
[]

79
[]

85
[]

60
[]

74
[]

27
[]

155
[]

132
[]

204
[]

176
[]

207
[]

121
[]

123
[]

98
[]

203
[]

134
[]

114
[]

104
[]

93
[2]

199
[31]

25
[]

193
[]

7
[]

178
[]

192
[]

50
[]

66
[]

8
[]

36
[]

182
[]

94
[]

30
[73]

45
[]

13
[5]

156
[53]

147
[]

187
[]

126
[]

86
[]

62
[]

112
[]

195
[]

125
[]

165
[]

189
[17]

191
[]

26
[]

95
[]

206
[]

88
[]

124
[11]

51
[]

162
[]

0
[]

131
[]

23
[]

37
[]

141
[]

89
[]

140
[]

9
[]

157
[]

105
[]

173
[]

128
[]

96
[]

122
[]

70
[]

3
[]

57
[]

146
[]

172
[]

78
[]

33
[]

174
[]

185
[29, 19]

197
[7]

29
[]

118
[]

38
[]

11
[]

4
[]

6
[]

49
[]

56
[]

181
[]

201
[]

209
[]

61
[]

198
[]

68
[]

183
[]

202
[]

92
[]

81
[]

154
[]

186
[]

110
[29]

91
[]

31
[]

73
[]


In [257]:
outputs_to_multiply = [0,1,2]
math.prod(dict_output[num_output] for num_output in outputs_to_multiply)

7847